# 문서/메모리 VLM 실습

## 목표

- 긴 OCR 출력에서 KV 캐시가 커지는 이유를 계산합니다.
- OCR plain text와 레이아웃 토큰의 차이를 비교합니다.
- 질문에 맞는 페이지 구간만 선택하는 Chain of Scroll toy algorithm을 구현합니다.


In [ ]:
def cache_tokens_full(reference_tokens, generated_tokens):
    return reference_tokens + generated_tokens

def cache_tokens_rswa(reference_tokens, generated_tokens, output_window):
    return reference_tokens + min(generated_tokens, output_window)

for generated in [256, 1024, 4096, 16384]:
    full = cache_tokens_full(512, generated)
    bounded = cache_tokens_rswa(512, generated, 256)
    print(f"generated={generated:>5} full={full:>6} rswa_like={bounded:>5}")


In [ ]:
page_tokens = [
    {"text": "E=mc^2", "row": 2, "col": 1},
    {"text": "diagram", "row": 2, "col": 12},
    {"text": "Table", "row": 5, "col": 1},
    {"text": "2026", "row": 6, "col": 4},
    {"text": "18", "row": 6, "col": 12},
]

plain_text = " ".join(token["text"] for token in page_tokens)
print("plain:", plain_text)
print("layout-aware:")
for token in page_tokens:
    print(token)


In [ ]:
pages = [
    {"page": 1, "keywords": {"abstract", "method"}},
    {"page": 2, "keywords": {"equation", "diagram"}},
    {"page": 3, "keywords": {"table", "benchmark"}},
    {"page": 4, "keywords": {"appendix", "license"}},
]

def chain_of_scroll(question, pages, max_steps=2):
    """질문 키워드와 많이 겹치는 페이지만 선택합니다."""
    q = set(question.lower().split())
    scored = []
    for page in pages:
        scored.append((len(q & page["keywords"]), page["page"]))
    return [page for _, page in sorted(scored, reverse=True)[:max_steps]]

print(chain_of_scroll("benchmark table result", pages))
print(chain_of_scroll("equation diagram layout", pages))
